# DSCI 551 Final Project — College Scorecard Database

Executes the full pipeline end to end: raw Scorecard CSVs → cleaned tables →
MySQL → workload queries and index experiments → Firestore subset →
interpretation.

Reusable logic lives in `src/`; this notebook stays focused on **execution,
evidence, and interpretation**.

*Team: <member 1>, <member 2>*

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebook":  # allow running with notebook/ as the working directory
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src import config, load_mysql, query_mysql, transform

## 1. Data

Copy the team's shared CSVs from Google Drive into `data/raw/` (gitignored),
then adjust the file names below to match the release we chose.

In [ ]:
institutions_raw = pd.read_csv(config.DATA_RAW_DIR / "MERGED2023-24_PP.csv", low_memory=False)
fields_raw = pd.read_csv(config.DATA_RAW_DIR / "FieldOfStudy2023-24.csv", low_memory=False)

institutions = transform.clean_institutions(institutions_raw)
fields_of_study = transform.clean_fields_of_study(fields_raw)

# Empty lists mean every canonical column found a source match in this release.
transform.missing_source_columns(institutions_raw, transform.INSTITUTION_FIELDS), \
transform.missing_source_columns(fields_raw, transform.FIELD_OF_STUDY_FIELDS)

## 2. Integrity evidence

`clean_institutions` / `clean_fields_of_study` already validated natural-key
uniqueness (they raise on duplicates or NULL keys). Below: row counts and
NULL rates per key analytic column — these are the suppression rates we
report when interpreting results.

In [ ]:
summary = pd.DataFrame(
    {
        "rows": [len(institutions), len(fields_of_study)],
        "unique_keys": [
            bool(institutions["unit_id"].is_unique),
            bool(~fields_of_study.duplicated(["unit_id", "cip_code", "credential_level"]).any()),
        ],
    },
    index=["institutions", "fields_of_study"],
)
display(summary)

institutions[
    ["adm_rate", "sat_avg", "completion_rate_4yr", "median_debt", "median_earnings_10yr"]
].isna().mean().round(3)

## 3. Load into MySQL

Prerequisites: `sql/01_create_schema.sql` executed once, credentials
configured via environment variables or `.streamlit/secrets.toml`. Load
`institutions` before `fields_of_study` (foreign key dependency).

In [ ]:
load_mysql.load_dataframe(institutions, "institutions")
load_mysql.load_dataframe(fields_of_study, "fields_of_study")

## 4. Workload queries

The shared queries live in `src/query_mysql.py` (SQL copies in
`sql/03_workload_queries.sql`). Two of them, executed live:

In [ ]:
display(query_mysql.run_query(*query_mysql.top_fields_of_study_query(limit=15)))
display(query_mysql.run_query(*query_mysql.completion_by_control_query()))

## 5. Index experiments

Protocol in `sql/04_index_experiments.sql`: for each experiment, capture
`EXPLAIN` (plan) and `EXPLAIN ANALYZE` (actual timing) with the index dropped,
then re-created. Record whether the plan changed (scan → ref/range) and the
time delta; paste plans into `results/index_experiments.md` (gitignored) and
summarize the takeaway in section 7.

In [ ]:
explorer_sql, explorer_params = query_mysql.filtered_institutions_query(states=["CA"], control=1)

baseline = query_mysql.run_query("EXPLAIN ANALYZE " + explorer_sql, explorer_params)
display(baseline)

# After running the CREATE INDEX step in sql/04, re-run the same statement and
# compare the last column (actual time) and the access type per row.

## 6. Firestore subset

One document per institution with its top fields of study embedded —
denormalized on purpose so a client reads a full profile in a single fetch.
Requires `FIREBASE_SERVICE_ACCOUNT` or the `[firebase]` section of
`.streamlit/secrets.toml`.

In [ ]:
from src.load_firebase import build_documents, init_firebase, upload_documents

app = init_firebase()
docs = build_documents(limit=200)
uploaded = upload_documents(docs, app=app)
print(f"Uploaded {uploaded} institution documents to Firestore.")

## 7. Interpretation

Record the evidence-backed takeaways here:

- **Index findings:** which indexes changed the plan (scan → ref/range), and
  the measured time delta from `EXPLAIN ANALYZE` before/after.
- **Data quality:** suppression rates per column (section 2) and how they
  constrain what we can claim.
- **MySQL vs Firestore:** what each store is good for in this project
  (relational filters + aggregations vs. single-document profile reads).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plot_df = institutions.dropna(subset=["completion_rate_4yr", "control"]).copy()
plot_df["ownership"] = plot_df["control"].map(query_mysql.CONTROL_LABELS)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(
    data=plot_df,
    x="ownership",
    y="completion_rate_4yr",
    order=["Public", "Private nonprofit", "Private for-profit"],
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("Completion rate (150% time)")
fig.tight_layout()
fig.savefig(FIGURES / "completion_by_control.png", dpi=150, bbox_inches="tight")
plt.show()